In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Trỏ thẳng vào thư mục repo đã có sẵn trên Drive
%cd /content/drive/MyDrive
!git clone https://github.com/fyb174147/forecast-models-evaluation.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/forecast-models-evaluation

# Môi trường chạy
- Chạy trên google colab 2026.07 (python 3.13)
- Các model không tương thích tạo 1 environment thông qua condacolab
- Cấu hình: 50GB RAM | GPU T4 (16GB) / L4 (24GB)
    - GPU L4 có tốc độ nhanh hơn 1.5-2x T4 tùy trường hợp
- Lưu ý: colab sẽ tự ngắt kết nối môi trường chạy nếu không sử dụng GPU trong 1-2h hoặc môi trường chạy đã online quá 24h. Trạng thái của môi trường thời gian chạy này, bao gồm tất cả các biến và tệp cục bộ, sẽ bị mất. Nên chia ra chạy từng đợt

# Khảo sát dữ liệu

In [ ]:
import os
import pandas as pd

DATA_DIR = "data"

datasets_to_inspect = {
    # 1. BELGIUM DATASET - Đợt 1 (2024-01-01 đến 2024-04-01: 3 tháng đầu năm)
    "Belgium - House 1 Solar (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_1/2024-01-01_0000_2024-04-01_0000/solar.csv",
    "Belgium - House 1 Battery (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_1/2024-01-01_0000_2024-04-01_0000/battery.csv",
    "Belgium - House 2 Solar (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_2/2024-01-01_0000_2024-04-01_0000/solar.csv",
    "Belgium - House 2 Battery (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_2/2024-01-01_0000_2024-04-01_0000/battery.csv",
    "Belgium - House 3 Solar (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_3/2024-01-01_0000_2024-04-01_0000/solar.csv",
    "Belgium - House 3 Battery (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_3/2024-01-01_0000_2024-04-01_0000/battery.csv",
    "Belgium - House 4 Solar (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_4/2024-01-01_0000_2024-04-01_0000/solar.csv",
    "Belgium - House 4 Battery (Jan-Apr 2024)": f"{DATA_DIR}/belgium_dataset/house_4/2024-01-01_0000_2024-04-01_0000/battery.csv",

    # 2. BELGIUM DATASET - Đợt 2 (2024-04-08 đến 2024-06-17: giai đoạn hè)
    "Belgium - House 1 Solar (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_1/2024-04-08_1500_2024-06-17_1500/solar.csv",
    "Belgium - House 1 Battery (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_1/2024-04-08_1500_2024-06-17_1500/battery.csv",
    "Belgium - House 2 Solar (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_2/2024-04-08_1500_2024-06-17_1500/solar.csv",
    "Belgium - House 2 Battery (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_2/2024-04-08_1500_2024-06-17_1500/battery.csv",
    "Belgium - House 3 Solar (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_3/2024-04-08_1500_2024-06-17_1500/solar.csv",
    "Belgium - House 3 Battery (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_3/2024-04-08_1500_2024-06-17_1500/battery.csv",
    "Belgium - House 4 Solar (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_4/2024-04-08_1500_2024-06-17_1500/solar.csv",
    "Belgium - House 4 Battery (Apr-Jun 2024)": f"{DATA_DIR}/belgium_dataset/house_4/2024-04-08_1500_2024-06-17_1500/battery.csv",

    # 3. GERMANY WPuQ DATASET (SFH 19 Load)
    "Germany - SFH 19 (3-month 2023-2024 benchmark)": f"{DATA_DIR}/germany_wpuq_dataset/SFH19_2023_2024_15min_3_month.csv",
    "Germany - SFH 19 (Original 2018-2019 1-year)": f"{DATA_DIR}/germany_wpuq_dataset/SFH19_2018_2019_15min_original.csv",

    # 4. LONDON LCL DATASET (LCL 33 Household Load)
    "London - LCL 33 (Cleaned Benchmark 2013)": f"{DATA_DIR}/london_dataset/LCL_london_consumption_2013.csv",
    "London - LCL 33 (Original Raw 2013)": f"{DATA_DIR}/london_dataset/LCL_london_consumption_2013_original.csv",

    # 5. ZONNEDAEL DATASET (Hà Lan - Khách hàng 8, 9, 43...)
    "Zonnedael - Household Customers (Liander 2013)": f"{DATA_DIR}/zonnedael_dataset/liander_zonnedael_2013_original.csv",
}

for name, path in datasets_to_inspect.items():
    if not os.path.exists(path):
        continue
    
    df = pd.read_csv(path)
    
    print("\n" + "="*80)
    print(f"{name.upper()} (File: {os.path.basename(path)})")
    print("="*80)
    
    # df.info()
    print("\n--- [df.info()] ---")
    df.info()
    
    # df.head()
    print("\n--- [df.head()] ---")
    display(df.head(3))

# Tiền xử lí dữ liệu
- Phần tiền xử lí dữ liệu đã được tác giả xử lí tại `utils/dataset_config.py`
- Mỗi khi chạy 1 model việc đầu tiên nó sẽ tự gọi việc tiền xử lí dữ liệu này

# 1. `model_naivedrift.py`

- Model nhẹ, có thể chạy trên CPU

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib
!python models/model_naivedrift.py

# 2. `models_statsml.py`
- Model nhẹ, có thể chạy trên CPU

In [ ]:
!pip install -q statsmodels pmdarima lightgbm xgboost scikit-learn
!python models/models_statsml.py

# 3. `models_neuralforecast.py`
- Bắt buộc dùng GPU có support CUDA core
- Chạy rất lâu và cần nhiều tài nguyên GPU
- Tổng thời gian chạy: 29h với GPU L4 (nhanh hơn khoảng 1.5-2x so với GPU T4)

In [ ]:
!pip install -q neuralforecast pytorch-lightning

# cờ --gpu = [0, -1] tương ứng dùng [GPU, CPU]
# data 1-bỉ, 2-đức, 3-london, 4-zonnedael
# sampling_rate 25, 33.3, 50, 100

!python models/models_neuralforecast.py --gpu 0 --data 1 --sampling_rate 25
!python models/models_neuralforecast.py --gpu 0 --data 1 --sampling_rate 33.333333
!python models/models_neuralforecast.py --gpu 0 --data 1 --sampling_rate 50
!python models/models_neuralforecast.py --gpu 0 --data 1 --sampling_rate 100

!python models/models_neuralforecast.py --gpu 0 --data 2 --sampling_rate 25
!python models/models_neuralforecast.py --gpu 0 --data 2 --sampling_rate 33.333333
!python models/models_neuralforecast.py --gpu 0 --data 2 --sampling_rate 50
!python models/models_neuralforecast.py --gpu 0 --data 2 --sampling_rate 100

!python models/models_neuralforecast.py --gpu 0 --data 3 --sampling_rate 25
!python models/models_neuralforecast.py --gpu 0 --data 3 --sampling_rate 33.333333
!python models/models_neuralforecast.py --gpu 0 --data 3 --sampling_rate 50
!python models/models_neuralforecast.py --gpu 0 --data 3 --sampling_rate 100

!python models/models_neuralforecast.py --gpu 0 --data 4 --sampling_rate 25
!python models/models_neuralforecast.py --gpu 0 --data 4 --sampling_rate 33.333333
!python models/models_neuralforecast.py --gpu 0 --data 4 --sampling_rate 50
!python models/models_neuralforecast.py --gpu 0 --data 4 --sampling_rate 100

# 4. `models_gluonts.py`
- Không tương thích với python 3.13, yêu cầu python 3.7. tạo 1 môi trường python 3.7 qua condacolab
- Có thể chạy với CPU, nhưng mất nhiều thời gian chạy kể cả khi chạy với GPU
- Tổng thời gian chạy: 35h với GPU T4 (không tương thích với GPU L4)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!conda create -n gluonts_gpu python=3.7 cudatoolkit=11.2 cudnn=8.1.0 -c conda-forge -y
!source activate gluonts_gpu && pip install mxnet-cu112==1.9.1 gluonts==0.14.4 pandas==1.3.5 numpy==1.21.6 ujson matplotlib scikit-learn
!source activate gluonts_gpu && pip install seaborn==0.12.2 scipy==1.7.3
!source activate gluonts_gpu && pip install matplotlib-inline

In [ ]:
# Sửa lỗi force=True không tương thích với Python 3.7
gluonts_file = "models/models_gluonts.py"

with open(gluonts_file, "r", encoding="utf-8") as f:
    content = f.read()

# Xóa bỏ dòng force=True
new_content = content.replace("        force=True\n", "").replace("        force=True,\n", "")

with open(gluonts_file, "w", encoding="utf-8") as f:
    f.write(new_content)

In [ ]:
import subprocess
import sys

cmd = """
source activate gluonts_gpu
export LD_LIBRARY_PATH=/usr/local/envs/gluonts_gpu/lib:$LD_LIBRARY_PATH
export MPLBACKEND=Agg

python -u models/models_gluonts.py --gpu 0 --data 1 --sampling_rate 25
python -u models/models_gluonts.py --gpu 0 --data 1 --sampling_rate 33.333333
python -u models/models_gluonts.py --gpu 0 --data 1 --sampling_rate 50
python -u models/models_gluonts.py --gpu 0 --data 1 --sampling_rate 100

python -u models/models_gluonts.py --gpu 0 --data 2 --sampling_rate 25
python -u models/models_gluonts.py --gpu 0 --data 2 --sampling_rate 33.333333
python -u models/models_gluonts.py --gpu 0 --data 2 --sampling_rate 50
python -u models/models_gluonts.py --gpu 0 --data 2 --sampling_rate 100

python -u models/models_gluonts.py --gpu 0 --data 3 --sampling_rate 25
python -u models/models_gluonts.py --gpu 0 --data 3 --sampling_rate 33.333333
python -u models/models_gluonts.py --gpu 0 --data 3 --sampling_rate 50
python -u models/models_gluonts.py --gpu 0 --data 3 --sampling_rate 100

python -u models/models_gluonts.py --gpu 0 --data 4 --sampling_rate 25
python -u models/models_gluonts.py --gpu 0 --data 4 --sampling_rate 33.333333
python -u models/models_gluonts.py --gpu 0 --data 4 --sampling_rate 50
python -u models/models_gluonts.py --gpu 0 --data 4 --sampling_rate 100
"""

# Online stream như chạy colab thông thường do condacolab không support online stream

process = subprocess.Popen(
    ["bash", "-c", cmd],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()

process.stdout.close()
process.wait()

# 5. `model_mamba.py`
- Không tương thích với python 3.13, hạ cấp về python 3.11
- Có thể hạ cấp phiên bản colab 2025.07 (python 3.11) và chạy trực tiếp
- Tổng thời gian: 8h với GPU L4

In [ ]:
!pip install -q torch==2.3.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install --no-deps https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0+cu122torch2.3cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
!pip install --no-deps https://github.com/state-spaces/mamba/releases/download/v2.2.2/mamba_ssm-2.2.2+cu122torch2.3cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

In [ ]:
!python models/model_mamba.py --gpu 0 --data 1 --sampling_rate 25
!python models/model_mamba.py --gpu 0 --data 1 --sampling_rate 33.333333
!python models/model_mamba.py --gpu 0 --data 1 --sampling_rate 50
!python models/model_mamba.py --gpu 0 --data 1 --sampling_rate 100

!python models/model_mamba.py --gpu 0 --data 2 --sampling_rate 25
!python models/model_mamba.py --gpu 0 --data 2 --sampling_rate 33.333333
!python models/model_mamba.py --gpu 0 --data 2 --sampling_rate 50
!python models/model_mamba.py --gpu 0 --data 2 --sampling_rate 100

!python models/model_mamba.py --gpu 0 --data 3 --sampling_rate 25
!python models/model_mamba.py --gpu 0 --data 3 --sampling_rate 33.333333
!python models/model_mamba.py --gpu 0 --data 3 --sampling_rate 50
!python models/model_mamba.py --gpu 0 --data 3 --sampling_rate 100

!python models/model_mamba.py --gpu 0 --data 4 --sampling_rate 25
!python models/model_mamba.py --gpu 0 --data 4 --sampling_rate 33.333333
!python models/model_mamba.py --gpu 0 --data 4 --sampling_rate 50
!python models/model_mamba.py --gpu 0 --data 4 --sampling_rate 100

# 6. `fmodel_chronos.py`

In [ ]:
!pip install -q chronos-forecasting

In [ ]:
!python models/fmodel_chronos.py --gpu 0 --data 1 --sampling_rates 25
!python models/fmodel_chronos.py --gpu 0 --data 1 --sampling_rates 33.333333
!python models/fmodel_chronos.py --gpu 0 --data 1 --sampling_rates 50
!python models/fmodel_chronos.py --gpu 0 --data 1 --sampling_rates 100

!python models/fmodel_chronos.py --gpu 0 --data 2 --sampling_rates 25
!python models/fmodel_chronos.py --gpu 0 --data 2 --sampling_rates 33.333333
!python models/fmodel_chronos.py --gpu 0 --data 2 --sampling_rates 50
!python models/fmodel_chronos.py --gpu 0 --data 2 --sampling_rates 100

!python models/fmodel_chronos.py --gpu 0 --data 3 --sampling_rates 25
!python models/fmodel_chronos.py --gpu 0 --data 3 --sampling_rates 33.333333
!python models/fmodel_chronos.py --gpu 0 --data 3 --sampling_rates 50
!python models/fmodel_chronos.py --gpu 0 --data 3 --sampling_rates 100

!python models/fmodel_chronos.py --gpu 0 --data 4 --sampling_rates 25
!python models/fmodel_chronos.py --gpu 0 --data 4 --sampling_rates 33.333333
!python models/fmodel_chronos.py --gpu 0 --data 4 --sampling_rates 50
!python models/fmodel_chronos.py --gpu 0 --data 4 --sampling_rates 100

# 7. `fmodel_timesfm.py`
- Không tương thích python 3.13, tạo môi trường python 3.11 qua condacolab

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!conda create -y -n timesfm_env python=3.11

!source activate timesfm_env && pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!source activate timesfm_env && pip install -q "timesfm[torch]==1.3.0"
!source activate timesfm_env && pip install matplotlib seaborn

In [ ]:
import subprocess
import sys

cmd = """
source activate timesfm_env
export MPLBACKEND=Agg

python -u models/fmodel_timesfm.py --gpu 0
"""

process = subprocess.Popen(
    ["bash", "-c", cmd],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()

process.stdout.close()
process.wait()

# 8. `fmodel_moirai.py`
- Không tương thích với python 3.13, tạo môi trường python 3.11 qua condacolab

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
%%bash
conda create -y -n moirai_env python=3.11
source activate moirai_env

pip install "torch==2.4.1+cu121" "torchvision==0.19.1+cu121" "torchaudio==2.4.1+cu121" --index-url https://download.pytorch.org/whl/cu121
pip install "numpy==1.26.4" "scipy==1.11.4" "fsspec<=2023.10.0" "pandas<2.2.0"
pip install --no-deps "gluonts>=0.14.3,<0.15" "uni2ts==2.0.0"
pip install \
    "jaxtyping==0.2.28" \
    "jax==0.4.35" \
    "jaxlib==0.4.35" \
    "datasets~=2.17.1" \
    "einops==0.7.0" \
    "hydra-core==1.3.2" \
    "python-dotenv==1.0.0" \
    "toolz<1.0.0,>=0.10.0" \
    lightning torchmetrics safetensors tensorboard multiprocess orjson ujson \
    pydantic tqdm typing-extensions matplotlib seaborn scikit-learn huggingface-hub

pip install "pandas==2.1.4" "numpy==1.26.4"

In [ ]:
import subprocess
import sys

cmd = """
source activate moirai_env
export MPLBACKEND=Agg

python -u models/fmodel_moirai.py --gpu 0 --data 1
python -u models/fmodel_moirai.py --gpu 0 --data 2
python -u models/fmodel_moirai.py --gpu 0 --data 3
python -u models/fmodel_moirai.py --gpu 0 --data 4
"""

process = subprocess.Popen(
    ["bash", "-c", cmd],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()

process.stdout.close()
process.wait()

# 9. `fmodel_timerxl.py`
- Không tương thích với python 3.13, tạo môi trường python 3.11 qua condacolab

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!conda create -y -n timerxl_env python=3.11

!source activate timerxl_env && conda install -y -c pytorch -c nvidia pytorch torchvision pytorch-cuda=12.1
!source activate timerxl_env && pip install "transformers==4.40.1" accelerate safetensors huggingface-hub pandas numpy matplotlib seaborn scikit-learn
!source activate timerxl_env && pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import subprocess
import sys

cmd = """
source activate timerxl_env
export MPLBACKEND=Agg

python -u models/fmodel_timerxl.py --gpu 0 --data 1
python -u models/fmodel_timerxl.py --gpu 0 --data 2
python -u models/fmodel_timerxl.py --gpu 0 --data 3
python -u models/fmodel_timerxl.py --gpu 0 --data 4
"""

process = subprocess.Popen(
    ["bash", "-c", cmd],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()

process.stdout.close()
process.wait()

# 10. `fmodel_timegpt.py`
- Yêu cầu timegpt api key, lấy tại https://www.nixtla.io/dashboard/api-keys
- Cần thêm api key vào trong colab

In [ ]:
!pip install -q nixtla python-dotenv

In [ ]:
import os
from google.colab import userdata

os.environ["TIMEGPT_API_KEY"] = userdata.get("TIMEGPT_API_KEY")

!python models/fmodel_timegpt.py --dataset belgium
!python models/fmodel_timegpt.py --dataset germany
!python models/fmodel_timegpt.py --dataset london
!python models/fmodel_timegpt.py --dataset zonnedael

# 11. Tổng hợp kết quả thực nghiệm thành bảng CSV chuẩn bài báo
- Tự động quét toàn bộ thư mục `results/` gom tất cả `model_metrics_summary.csv`
- Tính trung bình qua 10 lần chạy (runs) cho các mô hình Deep Learning
- Xuất ra 6 bảng riêng biệt (Table A.4 -> A.9) và 1 bảng gộp duy nhất

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

RESULTS_DIR = "results"
OUTPUT_DIR = "tables/experiment_tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NAME_MAP = {
    'BiTCN': 'BiTCN', 'Chronos': 'Chronos', 'DeepAR': 'DeepAR', 'DeepFactor': 'DeepFactor',
    'DeepNPTS': 'DeepNPTS', 'Informer': 'Informer', 'MOIRAI': 'MOIRAI', 'MQCNN': 'MQ-CNN',
    'MQRNN': 'MQ-RNN', 'Mamba': 'Mamba', 'NBEATS': 'N-BEATS', 'NHITS': 'NHITS',
    'NLinear': 'NLinear', 'PatchTST': 'PatchTST', 'TCN': 'TCN', 'TemporalFusionTransformer': 'TFT',
    'TiDE': 'TiDE', 'TimeGPT': 'TimeGPT', 'TimeXer': 'TimeXer', 'TimerXL': 'Timer-XL',
    'TimesFM': 'TimesFM', 'TimesNet': 'TimesNet', 'VanillaTransformer': 'Vanilla Transformer',
    'WaveNet': 'WaveNet', 'iTransformer': 'iTransformer', 'NaiveDrift': 'Naive Drift',
    'NaiveMovingAverage': 'Naive MA', 'LightGBM': 'LightGBM', 'KNNRegression': 'KNN Regression',
    'KNN': 'KNN Regression', 'ARIMA': 'AutoARIMA', 'AutoARIMA': 'AutoARIMA'
}

SERIES_MAP = {
    'pv_house_1': 'PV 1', 'pv_house_2': 'PV 2', 'pv_house_3': 'PV 3', 'pv_house_4': 'PV 4',
    'bess_house_1': 'BESS 1', 'bess_house_2': 'BESS 2', 'bess_house_3': 'BESS 3', 'bess_house_4': 'BESS 4',
    'germany_load': 'SFH 19', 'london_load': 'LCL 33',
    'zonnedael_customer_8': 'ZHC 8', 'zonnedael_customer_9': 'ZHC 9', 'zonnedael_customer_43': 'ZHC 43',
    'zonnedael_load_8': 'ZHC 8', 'zonnedael_load_9': 'ZHC 9', 'zonnedael_load_43': 'ZHC 43'
}

SAMPLING_MAP = {'Sampling_100': '100%', 'Sampling_50': '50%', 'Sampling_33': '33.33%', 'Sampling_25': '25%'}
SERIES_ORDER = ['SFH 19', 'LCL 33', 'ZHC 8', 'ZHC 9', 'ZHC 43', 'PV 1', 'PV 2', 'PV 3', 'PV 4', 'BESS 1', 'BESS 2', 'BESS 3', 'BESS 4']
SAMPLING_ORDER = ['100%', '50%', '33.33%', '25%']

TABLE_GROUPS = {
    'table_a4_statistical_ml': ['KNN Regression', 'LightGBM', 'AutoARIMA', 'Naive Drift', 'Naive MA'],
    'table_a5_mlp': ['N-BEATS', 'NHITS', 'TiDE', 'NLinear'],
    'table_a6_recurrent_gluonts': ['DeepAR', 'DeepFactor', 'MQ-RNN', 'Mamba', 'DeepNPTS'],
    'table_a7_convolutional': ['TCN', 'BiTCN', 'MQ-CNN', 'WaveNet', 'TimesNet'],
    'table_a8_transformer': ['Vanilla Transformer', 'Informer', 'PatchTST', 'TFT', 'TimeXer', 'iTransformer'],
    'table_a9_foundation_models': ['TimeGPT', 'Chronos', 'TimesFM', 'MOIRAI', 'Timer-XL']
}

all_summary_files = glob.glob(os.path.join(RESULTS_DIR, '**', 'model_metrics_summary.csv'), recursive=True)
records = []
for file_path in all_summary_files:
    if 'Epochs_1_' in file_path or 'Epochs_2_' in file_path:
        continue
    parts = os.path.relpath(file_path, RESULTS_DIR).split(os.sep)
    if len(parts) >= 5:
        dataset, raw_model, raw_sampling = parts[0], parts[1], parts[2]
        clean_model = NAME_MAP.get(raw_model, raw_model)
        clean_sampling = SAMPLING_MAP.get(raw_sampling)
        if not clean_sampling:
            continue
        try:
            df_metric = pd.read_csv(file_path)
            for _, row in df_metric.iterrows():
                clean_series = SERIES_MAP.get(str(row['model']).lower().strip())
                if clean_series:
                    records.append({
                        'Model': clean_model, 'Sampling': clean_sampling, 'Series': clean_series,
                        'MAE': float(row['MAE']), 'RMSE': float(row['RMSE'])
                    })
        except:
            pass

df_raw = pd.DataFrame(records)
df_agg = df_raw.groupby(['Model', 'Sampling', 'Series'])[['MAE', 'RMSE']].mean().reset_index()

all_table_dfs = []
for tbl_key, models_in_tbl in TABLE_GROUPS.items():
    formatted_rows = []
    for model in models_in_tbl:
        for samp in SAMPLING_ORDER:
            sub = df_agg[(df_agg['Model'] == model) & (df_agg['Sampling'] == samp)]
            if sub.empty:
                continue
            row_mae = {'Model': model, 'Sampling': samp, 'Metric': 'MAE'}
            for s in SERIES_ORDER:
                val = sub[sub['Series'] == s]['MAE'].values
                row_mae[s] = round(val[0], 3) if len(val) > 0 else np.nan
            formatted_rows.append(row_mae)
            row_rmse = {'Model': model, 'Sampling': samp, 'Metric': 'RMSE'}
            for s in SERIES_ORDER:
                val = sub[sub['Series'] == s]['RMSE'].values
                row_rmse[s] = round(val[0], 3) if len(val) > 0 else np.nan
            formatted_rows.append(row_rmse)
    df_tbl = pd.DataFrame(formatted_rows)
    df_tbl.to_csv(os.path.join(OUTPUT_DIR, f'{tbl_key}.csv'), index=False)
    if not df_tbl.empty:
        all_table_dfs.append(df_tbl)
    print(f'Saved {tbl_key}.csv: {len(df_tbl)} rows')

if all_table_dfs:
    df_consolidated = pd.concat(all_table_dfs, ignore_index=True)
    df_consolidated.to_csv(os.path.join(OUTPUT_DIR, 'all_tables_a4_to_a9_consolidated.csv'), index=False)
    print(f'Saved all_tables_a4_to_a9_consolidated.csv ({len(df_consolidated)} rows)!')


# 12. Vẽ biểu đồ cột so sánh Bài báo vs Thực nghiệm
- Mỗi mốc sampling gồm 4 cột: [Exp MAE, Exp RMSE, Paper MAE, Paper RMSE]
- Tự động lưu ảnh chất lượng cao vào `figures/comparisons/<Model_Name>/`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PAPER_CSV = 'tables/paper_tables/all_tables_a4_to_a9_consolidated.csv'
EXP_CSV = 'tables/experiment_tables/all_tables_a4_to_a9_consolidated.csv'
BASE_FIGURES_DIR = 'figures/comparisons'
os.makedirs(BASE_FIGURES_DIR, exist_ok=True)

SELECTED_MODELS = 'all'      # 'all' hoặc danh sách cụ thể: ['Naive MA', 'PatchTST', ...]
SELECTED_SERIES = 'all'      # 'all' hoặc danh sách chuỗi cụ thể: ['SFH 19', 'ZHC 8', ...]

SAMPLINGS = ['25%', '33.33%', '50%', '100%']
DISPLAY_SAMPLINGS = ['25%', '33%', '50%', '100%']
ALL_13_SERIES = ['SFH 19', 'LCL 33', 'ZHC 8', 'ZHC 9', 'ZHC 43', 'PV 1', 'PV 2', 'PV 3', 'PV 4', 'BESS 1', 'BESS 2', 'BESS 3', 'BESS 4']

df_p = pd.read_csv(PAPER_CSV)
df_e = pd.read_csv(EXP_CSV)

available_models = sorted(list(set(df_e['Model'].unique()).intersection(set(df_p['Model'].unique()))))
models_to_run = available_models if SELECTED_MODELS == 'all' else [m for m in SELECTED_MODELS if m in available_models]
series_to_run = ALL_13_SERIES if SELECTED_SERIES == 'all' else SELECTED_SERIES

def plot_model_data_comparison(model_name, series_name, save_path=None, show_inline=False):
    exp_mae, exp_rmse, paper_mae, paper_rmse = [], [], [], []
    for s in SAMPLINGS:
        r_e_m = df_e[(df_e['Model'] == model_name) & (df_e['Sampling'] == s) & (df_e['Metric'] == 'MAE')]
        r_e_r = df_e[(df_e['Model'] == model_name) & (df_e['Sampling'] == s) & (df_e['Metric'] == 'RMSE')]
        r_p_m = df_p[(df_p['Model'] == model_name) & (df_p['Sampling'] == s) & (df_p['Metric'] == 'MAE')]
        r_p_r = df_p[(df_p['Model'] == model_name) & (df_p['Sampling'] == s) & (df_p['Metric'] == 'RMSE')]
        v_e_m = r_e_m[series_name].values[0] if (not r_e_m.empty and series_name in r_e_m.columns) else np.nan
        v_e_r = r_e_r[series_name].values[0] if (not r_e_r.empty and series_name in r_e_r.columns) else np.nan
        v_p_m = r_p_m[series_name].values[0] if (not r_p_m.empty and series_name in r_p_m.columns) else np.nan
        v_p_r = r_p_r[series_name].values[0] if (not r_p_r.empty and series_name in r_p_r.columns) else np.nan
        exp_mae.append(v_e_m); exp_rmse.append(v_e_r); paper_mae.append(v_p_m); paper_rmse.append(v_p_r)
    all_vals = [v for v in (exp_mae + exp_rmse + paper_mae + paper_rmse) if not np.isnan(v)]
    if not all_vals:
        return False
    fig, ax = plt.subplots(figsize=(11, 6.5))
    x = np.arange(len(SAMPLINGS))
    width = 0.18
    r1 = ax.bar(x - 1.5*width, exp_mae, width, label='Exp MAE', color='#2563eb', edgecolor='black', linewidth=0.7)
    r2 = ax.bar(x - 0.5*width, exp_rmse, width, label='Exp RMSE', color='#1d4ed8', edgecolor='black', linewidth=0.7)
    r3 = ax.bar(x + 0.5*width, paper_mae, width, label='Paper MAE', color='#f97316', edgecolor='black', linewidth=0.7, hatch='//')
    r4 = ax.bar(x + 1.5*width, paper_rmse, width, label='Paper RMSE', color='#c2410c', edgecolor='black', linewidth=0.7, hatch='\\\\')
    for rects in [r1, r2, r3, r4]:
        for r in rects:
            h = r.get_height()
            if not np.isnan(h) and h > 0:
                ax.annotate(f'{h:.3f}', xy=(r.get_x() + r.get_width()/2, h), xytext=(0, 3),
                            textcoords='offset points', ha='center', va='bottom', fontsize=8, rotation=40)
    max_val = max(all_vals)
    ax.set_ylim(0, max_val * 1.45)
    for i in range(len(SAMPLINGS)):
        m_e, m_p = exp_mae[i], paper_mae[i]
        r_e, r_p = exp_rmse[i], paper_rmse[i]
        txt_lines = []
        if not np.isnan(m_e) and not np.isnan(m_p) and m_p != 0:
            d_m = m_e - m_p; pct_m = (d_m / m_p) * 100
            txt_lines.append(f'ΔMAE:  {d_m:+.3f} ({pct_m:+.1f}%)')
        if not np.isnan(r_e) and not np.isnan(r_p) and r_p != 0:
            d_r = r_e - r_p; pct_r = (d_r / r_p) * 100
            txt_lines.append(f'ΔRMSE: {d_r:+.3f} ({pct_r:+.1f}%)')
        if txt_lines:
            curr_max = max([v for v in [m_e, m_p, r_e, r_p] if not np.isnan(v)])
            ax.text(x[i], curr_max * 1.08 + 0.005, '\n'.join(txt_lines), ha='center', va='bottom', fontsize=8,
                    fontweight='bold', bbox=dict(boxstyle='round,pad=0.35', facecolor='#f8fafc', edgecolor='#94a3b8', alpha=0.95))
    ax.set_title(f'{model_name} - {series_name}', fontsize=14, fontweight='bold', loc='left', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(DISPLAY_SAMPLINGS, fontsize=11, fontweight='bold')
    ax.set_ylabel('Giá trị Sai số (MAE / RMSE)', fontsize=11, fontweight='bold')
    ax.legend(loc='upper right', frameon=True, fontsize=9.5, ncol=2)
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300)
    if show_inline:
        plt.show()
    else:
        plt.close()
    return True

print(f'🚀 Đang xuất biểu đồ cho {len(models_to_run)} mô hình...')
for model in models_to_run:
    clean_m = model.replace(' ', '_').replace('-', '_')
    for s in series_to_run:
        clean_s = s.replace(' ', '_')
        out_p = os.path.join(BASE_FIGURES_DIR, clean_m, f'{clean_m}_{clean_s}.png')
        plot_model_data_comparison(model, s, save_path=out_p, show_inline=False)
print('🎉 Đã hoàn tất xuất toàn bộ biểu đồ vào figures/comparisons!')


# 13. Kiểm định ý nghĩa thống kê Diebold-Mariano (DM test) trên kết quả tự thực nghiệm
- So sánh thống kê từng cặp mô hình trên chuỗi sai số dự báo: $e_t = y_t - \hat{y}_t$
- Kiểm định giả thuyết một phía (One-sided Diebold-Mariano test):
  + $H_0$: Hai mô hình có độ chính xác dự báo tương đương ($\mathbb{E}[d_t] = 0$).
  + $H_1$: Mô hình ở trục hoành ($x$-axis) có sai số kỳ vọng nhỏ hơn có ý nghĩa thống kê so với mô hình ở trục tung ($y$-axis) ($\mathbb{E}[d_t] < 0$).
- Đánh giá đồng thời trên 2 hàm mất mát: MAE loss ($|e|$) và RMSE loss ($e^2$).
- Ngưỡng ý nghĩa: $p \le 0.05$ (vùng màu tím sẫm) khẳng định mô hình cột $x$ vượt trội có ý nghĩa thống kê; $p > 0.1$ (vùng màu vàng) thể hiện hai mô hình tương đương về mặt thống kê.
- Tự động quét toàn bộ tệp `*_forecast_vs_actual.csv` trong `results/`, căn chỉnh mốc thời gian chung và sinh bản đồ nhiệt ma trận p-value.
- Lưu trữ đồ thị chuẩn mực bài báo (PDF & PNG) tại `results/dm_test/` và hiển thị trực tiếp trên Notebook.

In [ ]:
# ==============================================================================
# 1. LỆNH CHẠY RIÊNG KIỂM ĐỊNH DIEBOLD-MARIANO LOCAL CHO CẢ 4 DATASETS:
# (Tự động tính toán & xuất toàn bộ bản đồ nhiệt PDF/PNG cho belgium, germany, london, zonnedael)
# ==============================================================================
!python utils/dm_test.py --datasets belgium germany london zonnedael --sampling all

# ==============================================================================
# 2. HOẶC CHẠY TOÀN BỘ PIPELINE (Bảng CSV + Biểu đồ cột + Kiểm định DM test cả 4 datasets):
# ==============================================================================
# !python run_all_evaluations.py


In [ ]:
# ==============================================================================
# CÁCH 2: Chạy trực tiếp Python, hiển thị bản đồ nhiệt và bảng p-value ngay trên Notebook
# ==============================================================================
import os
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import numpy as np
from utils.dm_test import (
    find_model_dirs,
    read_target_series,
    _compute_common_timestamp_index,
    _align_losses_to_index,
    _build_losses_from_actual_forecast,
    build_dm_matrix
)

DM_RESULTS_DIR = Path("results/dm_test/belgium")
os.makedirs(DM_RESULTS_DIR, exist_ok=True)

# 1. Hiển thị hình ảnh ma trận nhiệt DM test trực tiếp trên notebook
heatmaps = [
    ("dm_test_pv_house_1_samp25.png", "PV House 1 (Sampling 25% - Deep Learning Models)"),
    ("dm_test_bess_house_1_samp25.png", "BESS House 1 (Sampling 25% - Deep Learning Models)"),
    ("dm_test_pv_house_1_samp100.png", "PV House 1 (Sampling 100% - Foundation Models & Baselines)")
]

for fname, title in heatmaps:
    fpath = DM_RESULTS_DIR / fname
    if fpath.exists():
        plt.figure(figsize=(12, 6))
        img = mpimg.imread(str(fpath))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Diebold-Mariano Test Heatmap: {title}", fontsize=11, fontweight="bold", pad=12)
        plt.tight_layout()
        plt.show()

# 2. Xuất bảng định lượng p-value đối chiếu các mô hình SOTA vs NaiveDrift baseline
dataset_dir = Path("results/results_belgium")
model_dirs = find_model_dirs(dataset_dir)
act_map, fc_map = {}, {}
for md in model_dirs:
    a, f, _ = read_target_series(md, "PV_house_1_forecast_vs_actual.csv", "Sampling_25")
    if f is not None:
        act_map[md.name] = a
        fc_map[md.name] = f

c_idx = _compute_common_timestamp_index(fc_map)
act_map = _align_losses_to_index(act_map, c_idx)
fc_map = _align_losses_to_index(fc_map, c_idx)
mae_losses, rmse_losses = {}, {}
models = list(fc_map.keys())
for m in models:
    mae_losses[m], rmse_losses[m] = _build_losses_from_actual_forecast(act_map[m], fc_map[m])

df_mae = pd.DataFrame(build_dm_matrix(mae_losses, models), index=models, columns=models)
df_rmse = pd.DataFrame(build_dm_matrix(rmse_losses, models), index=models, columns=models)

summary_rows = []
for m in models:
    if m == "NaiveDrift": continue
    p_mae = df_mae.loc[m, "NaiveDrift"]
    p_rmse = df_rmse.loc[m, "NaiveDrift"]
    summary_rows.append({
        "Mô hình": m,
        "p-value (MAE loss)": f"{p_mae:.6f}",
        "p-value (RMSE loss)": f"{p_rmse:.6f}",
        "Ý nghĩa thống kê (alpha=0.05)": "Vượt trội có ý nghĩa (p < 0.05)" if p_mae <= 0.05 else "Tương đương (p > 0.05)"
    })

df_summary = pd.DataFrame(summary_rows)
print("\n" + "="*85)
print("BẢNG TỔNG HỢP KIỂM ĐỊNH DIEBOLD-MARIANO TỰ THỰC NGHIỆM (PV 1, Sampling 25%)")
print("Giả thuyết đối H1: Mô hình kiểm định có sai số thấp hơn NaiveDrift baseline")
print("="*85)
try:
    from IPython.display import display
    display(df_summary)
except Exception:
    print(df_summary.to_string(index=False))
